In [ ]:
import pandas as pd
from langchain_community.llms import Ollama
from langchain_experimental.agents import create_pandas_dataframe_agent
from langchain_community.embeddings import OllamaEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# 1. Data Preparation
df = pd.read_csv("product_data.csv")

# 2. Initialize Components
llm = Ollama(model="llama3.1", temperature=0.75)
embeddings = OllamaEmbeddings(
    model="bge-m3",
    base_url="http://localhost:11434"
)

# 3. Create Detailed Text Description
def create_detailed_text(df):
    texts = []
    for index, row in df.iterrows():
        text = f"Product Details - "
        for col in df.columns:
            text += f"{col}: {row[col]}. "
        texts.append(text)
    return texts

# 4. Prepare Documents
texts = create_detailed_text(df)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
)
splits = text_splitter.create_documents(texts)

# 5. Create Vector Store
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

# 6. Create Prompt Template
template = """
Answer the question based on the following information:

Context Information:
{context}

Question: {question}

If there is no relevant information in the data, please clearly state "No information about this product in the database".
Please answer based on actual data, do not guess or make up information.

Answer:
"""

PROMPT = PromptTemplate(
    template=template,
    input_variables=["context", "question"]  
)

# 7. Setup QA Chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
    chain_type_kwargs={
        "prompt": PROMPT
    }
)

# 8. Create Agent
agent = create_pandas_dataframe_agent(
    llm,
    df,
    verbose=True,
    allow_dangerous_code=True
)

# 9. Create Universal Query Function
def query_data(query_text):
    """
    Universal data query function
    Args:
        query_text: User's query question
    Returns:
        Query results and related context
    """
    try:
        # Use RAG system to get context
        context = qa_chain.invoke({
            "query": query_text
        })
        
        # Build enhanced prompt
        enhanced_prompt = f"""
        Answer the question based on the following information:
        1. Retrieved relevant information: {context['result']}
        2. Columns in the dataframe: {list(df.columns)}
        
        Question: {query_text}
        
        Please provide a detailed answer, if calculations or analysis are needed, please base them on the data.
        """
        
        # Use agent to process query
        response = agent.run(enhanced_prompt)
        
        return {
            "answer": response,
            "context": context['source_documents']
        }
    except Exception as e:
        return {
            "error": str(e),
            "suggestion": "Please check if the query is related to the dataset"
        }

# 10. Query Result Display Function
def display_query_result(query_text):
    """
    Function to display query results
    Args:
        query_text: User's query question
    """
    print(f"\nQuery: {query_text}")
    print("-" * 50)
    
    result = query_data(query_text)
    
    if "error" in result:
        print(f"Query Error: {result['error']}")
        print(f"Suggestion: {result['suggestion']}")
    else:
        print(f"Answer: {result['answer']}")
        print("\nRelated Context:")
        for doc in result['context']:
            print(f"- {doc.page_content}")
    print("-" * 50)

# Usage Examples
# You can perform various queries
queries = [
    "What is the most expensive product in the dataset?",
    "How many different product categories are there?",
    "What is the average price?",
    "Which product has the highest inventory?"
]

for query in queries:
    display_query_result(query)

